## 1. What is Class Imbalance?

**Class imbalance** happens when one class in your target has way more examples than another class.

- Example: 950 "normal" cases vs 50 "fraud" cases.
- The rare class (fraud) is usually the one we care most about detecting.

**Real-world example:** In fraud detection, ~99% of transactions are legit and ~1% are fraud.

**Business example:** In manufacturing quality control, most items pass inspection — defective items are rare.

**AI/ML use case:** In rare disease diagnosis, "positive" cases are far fewer than "negative" (healthy) cases in the training data.

In [ ]:
import pandas as pd

labels = ["legit"] * 18 + ["fraud"] * 2   # 18 legit, 2 fraud
df = pd.DataFrame({"transaction_type": labels})

counts = df["transaction_type"].value_counts()
print("Class counts:")
print(counts)

print("\nClass percentages:")
print((counts / len(df) * 100).round(1))

Class counts:
transaction_type
legit    18
fraud     2
Name: count, dtype: int64

Class percentages:
transaction_type
legit    90.0
fraud    10.0
Name: count, dtype: float64


## 2. Balanced vs Imbalanced Dataset

- **Balanced dataset:** classes have roughly equal numbers of examples.
- **Imbalanced dataset:** one class heavily outnumbers the other(s).

**Real-world example:** Balanced — coin flip results (~50% heads, ~50% tails). Imbalanced — spam emails (spam is much rarer than normal email).

**Business example:** Balanced — a survey with roughly equal male/female responses. Imbalanced — customer churn data, where churners are a small minority.

**AI/ML use case:** Balanced — handwritten digit recognition (roughly equal digits 0–9). Imbalanced — credit card fraud detection (fraud is rare).

In [ ]:
import pandas as pd

balanced = pd.Series(["yes"] * 10 + ["no"] * 10)
imbalanced = pd.Series(["yes"] * 18 + ["no"] * 2)

print("Balanced dataset counts:")
print(balanced.value_counts())

print("\nImbalanced dataset counts:")
print(imbalanced.value_counts())

Balanced dataset counts:
yes    10
no     10
Name: count, dtype: int64

Imbalanced dataset counts:
yes    18
no      2
Name: count, dtype: int64


## 3. Why Class Imbalance is a Problem (and why Accuracy Can Lie)

When one class dominates, a model can get **lazy**: it learns to just predict the majority class every time, and still score a high **accuracy** — while completely failing at the thing you actually care about (catching the rare class).

**Why accuracy is misleading here:**
Accuracy = (correct predictions) / (total predictions). If 95% of data is "legit", a model that *always* says "legit" gets 95% accuracy — but it catches **0%** of actual fraud. The number looks great, but the model is useless.

**Better metrics for imbalanced data:** Precision, Recall, F1-score, and the Confusion Matrix — they show performance *per class*, not just overall.

**Real-world example:** A disease-screening model that always predicts "healthy" looks 98% accurate but misses every real patient — dangerous.

**Business example:** A "99% accurate" fraud model that catches zero frauds saves the business nothing.

**AI/ML use case:** Any rare-event model (fraud, churn, defects) should be evaluated with recall/precision/F1, not accuracy alone.

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

# True labels: 18 legit (0), 2 fraud (1) -- imbalanced
y_true = [0]*18 + [1]*2

# A "lazy" model that always predicts "legit" (0)
y_pred_lazy = [0] * 20

print("Accuracy:", accuracy_score(y_true, y_pred_lazy))
print("Recall for fraud class:", recall_score(y_true, y_pred_lazy))
print("Precision for fraud class:", precision_score(y_true, y_pred_lazy, zero_division=0))
print("\nConfusion matrix:\n", confusion_matrix(y_true, y_pred_lazy))
print("\n-> 90% accuracy looks great, but recall = 0 -> it caught ZERO frauds!")

Accuracy: 0.9
Recall for fraud class: 0.0
Precision for fraud class: 0.0

Confusion matrix:
 [[18  0]
 [ 2  0]]

-> 90% accuracy looks great, but recall = 0 -> it caught ZERO frauds!


## 4. Undersampling

**Undersampling** = reduce the number of majority-class rows so it matches the minority class size.

- Pros: fast, simple.
- Cons: you throw away data, which can lose useful information.

**Real-world example:** Keeping all rare "network attack" logs but only a small sample of the huge number of "normal traffic" logs.

**Business example:** Keeping all "high-value complaint" cases but sampling only some "routine inquiry" cases for training.

**AI/ML use case:** Common first step in fraud-detection pipelines to balance a huge imbalanced training set.

In [2]:
import pandas as pd

df = pd.DataFrame({
    "feature1": [1,2,1,2,3,2,1,3,2,1, 8,9],
    "feature2": [1,1,2,2,1,3,3,2,1,2, 8,9],
    "target":   [0,0,0,0,0,0,0,0,0,0, 1,1],   # 10 majority (0), 2 minority (1)
})
print("Before undersampling:")
print(df["target"].value_counts())

majority = df[df["target"] == 0]
minority = df[df["target"] == 1]

majority_undersampled = majority.sample(n=len(minority), random_state=42)
balanced_df = pd.concat([majority_undersampled, minority])

print("\nAfter undersampling:")
print(balanced_df["target"].value_counts())

Before undersampling:
target
0    10
1     2
Name: count, dtype: int64

After undersampling:
target
0    2
1    2
Name: count, dtype: int64


## 5. Oversampling

**Oversampling** = increase the number of minority-class rows (by duplicating or generating new ones) so it matches the majority class size.

- Pros: no data thrown away.
- Cons: naive duplication can cause the model to overfit on the repeated examples.

**Real-world example:** Oversampling rare "disease-positive" patient records so the model sees enough examples.

**Business example:** Oversampling rare "churned customer" records so the model learns what churn looks like.

**AI/ML use case:** Oversampling rare defect images so a quality-inspection model gets enough exposure to defects.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "feature1": [1,2,1,2,3,2,1,3,2,1, 8,9],
    "feature2": [1,1,2,2,1,3,3,2,1,2, 8,9],
    "target":   [0,0,0,0,0,0,0,0,0,0, 1,1],   # 10 majority (0), 2 minority (1)
})
print("Before oversampling:")
print(df["target"].value_counts())

majority = df[df["target"] == 0]
minority = df[df["target"] == 1]

minority_oversampled = minority.sample(n=len(majority), replace=True, random_state=42)
balanced_df = pd.concat([majority, minority_oversampled])

print("\nAfter oversampling:")
print(balanced_df["target"].value_counts())

Before oversampling:
target
0    10
1     2
Name: count, dtype: int64

After oversampling:
target
0    10
1    10
Name: count, dtype: int64


## 6. Random Oversampling

The simplest oversampling method: just **randomly duplicate** existing minority-class rows (with replacement) until the classes are balanced. (This is what `sample(replace=True)` did in the previous topic — it has a name: Random Oversampling.)

**Real-world example:** Duplicating existing photos of a rare bird species so the dataset has more "examples" of it (even though they're copies).

**Business example:** Duplicating rare "VIP customer" records in a customer-segmentation training set.

**AI/ML use case:** A quick baseline oversampling method, often tried before more advanced techniques like SMOTE.

In [ ]:
import pandas as pd
from sklearn.utils import resample

df = pd.DataFrame({
    "feature1": [1,2,1,2,3,2,1,3,2,1, 8,9],
    "target":   [0,0,0,0,0,0,0,0,0,0, 1,1],
})
majority = df[df["target"] == 0]
minority = df[df["target"] == 1]

minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=42)
balanced_df = pd.concat([majority, minority_upsampled])

print("Class counts after Random Oversampling:")
print(balanced_df["target"].value_counts())
print("\nNotice the minority rows are exact copies of the original 2 rows:")
print(minority_upsampled)

Class counts after Random Oversampling:
target
0    10
1    10
Name: count, dtype: int64

Notice the minority rows are exact copies of the original 2 rows:
    feature1  target
10         8       1
11         9       1
10         8       1
10         8       1
10         8       1
11         9       1
10         8       1
10         8       1
10         8       1
11         9       1


## 7. Random Undersampling

The simplest undersampling method: just **randomly remove** majority-class rows until the classes are balanced. (This is what `sample()` did in topic 4 — its name is Random Undersampling.)

**Real-world example:** Randomly keeping only some of millions of "normal" network logs to match the number of rare "intrusion" logs.

**Business example:** Randomly sampling a subset of common "standard" support tickets to match the rare "escalated" ticket count.

**AI/ML use case:** A quick baseline undersampling method, often the first thing tried in an imbalanced-learning pipeline.

In [ ]:
import pandas as pd
from sklearn.utils import resample

df = pd.DataFrame({
    "feature1": [1,2,1,2,3,2,1,3,2,1, 8,9],
    "target":   [0,0,0,0,0,0,0,0,0,0, 1,1],
})
majority = df[df["target"] == 0]
minority = df[df["target"] == 1]

majority_downsampled = resample(majority, replace=False, n_samples=len(minority), random_state=42)
balanced_df = pd.concat([majority_downsampled, minority])

print("Class counts after Random Undersampling:")
print(balanced_df["target"].value_counts())
print("\nOnly 2 of the original 10 majority rows were kept:")
print(majority_downsampled)

Class counts after Random Undersampling:
target
0    2
1    2
Name: count, dtype: int64

Only 2 of the original 10 majority rows were kept:
   feature1  target
8         2       0
1         2       0


## 8. SMOTE (Synthetic Minority Oversampling Technique)

Instead of just copying minority rows, **SMOTE creates new, synthetic minority examples** by picking a minority point, finding one of its nearest minority neighbors, and generating a new point *somewhere in between* them.

- This gives the model more variety than plain duplication (topic 6), so it's less likely to overfit on exact copies.
- In real projects you'd normally use the `imbalanced-learn` library (`from imblearn.over_sampling import SMOTE`). Below is a simplified, from-scratch version so you can see exactly how it works.

**Real-world example:** Generating synthetic "fraud" transactions that sit between two real fraud cases, instead of exact duplicates.

**Business example:** Generating new, slightly-varied synthetic "high-risk loan applicant" profiles for training.

**AI/ML use case:** Widely used on tabular/medical data to boost minority-class recall without the overfitting risk of plain duplication.

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

# A few minority-class points (2 features each)
minority_points = np.array([[8, 8], [9, 9], [8, 9]])

# Find each point's nearest minority neighbor
nn = NearestNeighbors(n_neighbors=2).fit(minority_points)
_, neighbor_idx = nn.kneighbors(minority_points)

synthetic_points = []
for i, point in enumerate(minority_points):
    neighbor = minority_points[neighbor_idx[i][1]]   # nearest neighbor (not itself)
    gap = neighbor - point
    new_point = point + np.random.rand() * gap        # random point BETWEEN them
    synthetic_points.append(new_point)

print("Original minority points:\n", minority_points)
print("\nNew SMOTE synthetic points (in between real points):")
print(np.round(synthetic_points, 2))

Original minority points:
 [[8 8]
 [9 9]
 [8 9]]

New SMOTE synthetic points (in between real points):
[[8.   8.53]
 [8.63 9.  ]
 [8.   8.61]]


## 9. Borderline-SMOTE

A variant of SMOTE that only generates synthetic points for minority examples sitting **near the class boundary** (surrounded by majority-class neighbors) — the "risky" points a model is most likely to get wrong. "Safe" minority points deep inside the minority region are mostly left alone.

**Real-world example:** Focusing synthetic examples on ambiguous disease cases that look similar to healthy patients (the hard cases), not the obvious ones.

**Business example:** Focusing on "at risk of churning but hasn't churned yet" customers — the borderline, hardest-to-classify group.

**AI/ML use case:** Often improves the decision boundary more than plain SMOTE, since it reinforces exactly the examples the model struggles with.

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

# Majority points (label 0) and minority points (label 1)
majority_points = np.array([[2,2],[2,3],[3,2],[3,3],[5,7]])
minority_points = np.array([[6,6],[8,8],[9,9]])   # [6,6] sits close to majority point [5,7]

all_points = np.vstack([majority_points, minority_points])
all_labels = np.array([0]*len(majority_points) + [1]*len(minority_points))

# For each minority point, look at its 2 nearest OTHER points (excluding itself)
nn = NearestNeighbors(n_neighbors=3).fit(all_points)
for point, label in zip(minority_points, ["minority_1", "minority_2", "minority_3"]):
    _, idx = nn.kneighbors([point])
    neighbor_idx = idx[0][1:]              # drop index 0 = the point itself (distance 0)
    neighbor_labels = all_labels[neighbor_idx]
    majority_neighbor_count = (neighbor_labels == 0).sum()
    status = "BORDERLINE (near majority)" if majority_neighbor_count >= 1 else "safe (deep in minority region)"
    print(f"{label} {point} -> {majority_neighbor_count}/2 majority neighbors -> {status}")

print("\n-> Borderline-SMOTE would generate new synthetic points mainly around the BORDERLINE ones.")

minority_1 [6 6] -> 1/2 majority neighbors -> BORDERLINE (near majority)
minority_2 [8 8] -> 0/2 majority neighbors -> safe (deep in minority region)
minority_3 [9 9] -> 0/2 majority neighbors -> safe (deep in minority region)

-> Borderline-SMOTE would generate new synthetic points mainly around the BORDERLINE ones.


## 10. Class Weights

Instead of changing the data at all, tell the model to **pay more attention** to the minority class by giving it a higher weight in the loss function (mistakes on it "cost" more).

- No data is added or removed — just an extra parameter.
- Very common in scikit-learn: `class_weight="balanced"`.

**Real-world example:** A fraud model penalizes missing real fraud (false negative) much more heavily than a false alarm.

**Business example:** A quality-control model penalizes missing a real defect more heavily than a false alarm, since a missed defect is costlier.

**AI/ML use case:** `class_weight` in scikit-learn or `scale_pos_weight` in XGBoost — a quick fix that needs no changes to the dataset itself.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score
import numpy as np

# Imbalanced 1-feature data: 18 majority points (0-6) and 2 minority points (6-7)
# Notice x=6 overlaps both classes -> a tricky, borderline point
majority_x = [1,1,1,2,2,2,3,3,3,4,4,4,5,5,5,6,6,6]
minority_x = [6, 7]
X = np.array(majority_x + minority_x).reshape(-1, 1)
y = np.array([0]*len(majority_x) + [1]*len(minority_x))

# Without class weights
model_default = LogisticRegression().fit(X, y)
recall_default = recall_score(y, model_default.predict(X))

# With class weights -> mistakes on the minority class cost more
model_weighted = LogisticRegression(class_weight="balanced").fit(X, y)
recall_weighted = recall_score(y, model_weighted.predict(X))

print(f"Recall on minority class WITHOUT class_weight: {recall_default:.2f}")
print(f"Recall on minority class WITH class_weight='balanced': {recall_weighted:.2f}")
print("\n-> class_weight helped the model correctly catch the borderline minority point too.")

Recall on minority class WITHOUT class_weight: 0.50
Recall on minority class WITH class_weight='balanced': 1.00

-> class_weight helped the model correctly catch the borderline minority point too.


## 11. When to Use Each Technique

Quick decision guide:

| Situation | Recommended technique |
|---|---|
| Dataset is small, can't afford to lose data | Oversampling / SMOTE |
| Dataset is huge, majority class is massive | Undersampling |
| Need a fast, simple baseline fix | Random Oversampling / Random Undersampling |
| Duplicated copies are causing overfitting | SMOTE |
| The hardest, borderline cases matter most | Borderline-SMOTE |
| Don't want to touch the data at all | Class Weights |

**Real-world example:** A hospital with very limited rare-disease records would prefer SMOTE over undersampling (can't afford to throw away the little majority data they have).

**Business example:** A huge transaction log (millions of "legit" rows) can safely use undersampling without losing much signal.

**AI/ML use case:** Many production pipelines start simple (`class_weight="balanced"`), and only move to SMOTE/Borderline-SMOTE if recall on the minority class is still too low.

In [ ]:
guide = {
    "Small dataset, can't lose data":        "Oversampling / SMOTE",
    "Huge dataset, massive majority class":  "Undersampling",
    "Need a fast, simple baseline":          "Random Oversampling / Random Undersampling",
    "Duplicates causing overfitting":        "SMOTE",
    "Borderline/hard cases matter most":     "Borderline-SMOTE",
    "Don't want to touch the data":          "Class Weights",
}

for situation, technique in guide.items():
    print(f"{situation:40s} -> {technique}")

Small dataset, can't lose data           -> Oversampling / SMOTE
Huge dataset, massive majority class     -> Undersampling
Need a fast, simple baseline             -> Random Oversampling / Random Undersampling
Duplicates causing overfitting           -> SMOTE
Borderline/hard cases matter most        -> Borderline-SMOTE
Don't want to touch the data             -> Class Weights
